In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/adityachandra1611/train-dataset/train.csv
/kaggle/input/datasets/adityachandra1611/sample-submission/sample_submission.csv
/kaggle/input/datasets/adityachandra1611/test-traffic/test.csv


In [2]:
import os
print(os.path.exists("/kaggle/working/submission.csv"))

False


In [3]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

# ==========================================
# 1. Load Data
# ==========================================

train = pd.read_csv("/kaggle/input/datasets/adityachandra1611/train-dataset/train.csv")
test = pd.read_csv("/kaggle/input/datasets/adityachandra1611/test-traffic/test.csv")

# ==========================================
# 2. Feature Engineering
# ==========================================

def add_features(df):
    df = df.copy()

    # --------------------------
    # Geohash Features
    # --------------------------
    if "geohash" in df.columns:
        df["geo4"] = df["geohash"].astype(str).str[:4]
        df["geo5"] = df["geohash"].astype(str).str[:5]
        df["geo6"] = df["geohash"].astype(str).str[:6]

    # --------------------------
    # Timestamp Features
    # --------------------------
    if "timestamp" in df.columns:

        ts = pd.to_datetime(
            df["timestamp"],
            errors="coerce"
        )

        if ts.notna().sum() > 0:

            df["hour"] = ts.dt.hour
            df["minute"] = ts.dt.minute
            df["weekday"] = ts.dt.dayofweek

            # Cyclical encoding
            df["hour_sin"] = np.sin(
                2 * np.pi * df["hour"] / 24
            )

            df["hour_cos"] = np.cos(
                2 * np.pi * df["hour"] / 24
            )

            df["weekday_sin"] = np.sin(
                2 * np.pi * df["weekday"] / 7
            )

            df["weekday_cos"] = np.cos(
                2 * np.pi * df["weekday"] / 7
            )

        df.drop(columns=["timestamp"], inplace=True)

    # --------------------------
    # Interaction Feature
    # --------------------------
    if (
        "RoadType" in df.columns
        and "Weather" in df.columns
    ):
        df["road_weather"] = (
            df["RoadType"].astype(str)
            + "_"
            + df["Weather"].astype(str)
        )

    # --------------------------
    # Day Numeric
    # --------------------------
    if "day" in df.columns:
        df["day"] = pd.to_numeric(
            df["day"],
            errors="coerce"
        )

    return df


# ==========================================
# 3. Prepare Data
# ==========================================

target = "demand"

X = train.drop(columns=[target])
y = train[target]

X_test = test.copy()

X = add_features(X)
X_test = add_features(X_test)

# Save IDs
test_ids = X_test["Index"]

# Remove ID
X.drop(columns=["Index"], inplace=True)
X_test.drop(columns=["Index"], inplace=True)

# Align columns
X_test = X_test.reindex(
    columns=X.columns,
    fill_value=np.nan
)

# ==========================================
# 4. Handle Missing Values
# ==========================================

cat_cols = [
    c for c in X.columns
    if X[c].dtype == "object"
]

for col in X.columns:

    if col in cat_cols:

        X[col] = (
            X[col]
            .fillna("missing")
            .astype(str)
        )

        X_test[col] = (
            X_test[col]
            .fillna("missing")
            .astype(str)
        )

    else:

        X[col] = pd.to_numeric(
            X[col],
            errors="coerce"
        )

        X_test[col] = pd.to_numeric(
            X_test[col],
            errors="coerce"
        )

        median = X[col].median()

        X[col] = X[col].fillna(median)
        X_test[col] = X_test[col].fillna(median)

# CatBoost categorical indices
cat_feature_indices = [
    X.columns.get_loc(c)
    for c in cat_cols
]

# ==========================================
# 5. Cross Validation
# ==========================================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(kf.split(X), 1):

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    train_pool = Pool(
        X_train,
        y_train,
        cat_features=cat_feature_indices
    )

    valid_pool = Pool(
        X_valid,
        y_valid,
        cat_features=cat_feature_indices
    )

    model = CatBoostRegressor(
        iterations=10000,
        learning_rate=0.02,
        depth=10,
        loss_function="RMSE",
        eval_metric="R2",
        random_seed=42,
        od_type="Iter",
        od_wait=500,
        l2_leaf_reg=5,
        verbose=500
    )

    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True
    )

    oof_preds[valid_idx] = model.predict(X_valid)

    test_preds += (
        model.predict(X_test)
        / kf.n_splits
    )

    score = r2_score(
        y_valid,
        oof_preds[valid_idx]
    )

    print(
        f"Fold {fold} R2 = {score:.6f}"
    )

# ==========================================
# 6. Overall Score
# ==========================================

overall_score = r2_score(
    y,
    oof_preds
)

print(
    "\nOverall CV R2:",
    overall_score
)

# ==========================================
# 7. Submission
# ==========================================

submission = pd.DataFrame({
    "Index": test_ids,
    "demand": test_preds
})

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print(submission.shape)
print(submission.head())

print(
    "\nSaved successfully:"
)

print(
    "/kaggle/working/submission.csv"
)

/tmp/ipykernel_16/688166259.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ts = pd.to_datetime(
/tmp/ipykernel_16/688166259.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ts = pd.to_datetime(


0:	learn: 0.0290093	test: 0.0296307	best: 0.0296307 (0)	total: 183ms	remaining: 30m 32s
500:	learn: 0.9221012	test: 0.9190744	best: 0.9190744 (500)	total: 57.7s	remaining: 18m 14s
1000:	learn: 0.9347352	test: 0.9256656	best: 0.9256670 (999)	total: 2m 1s	remaining: 18m 10s
1500:	learn: 0.9417834	test: 0.9280062	best: 0.9280066 (1499)	total: 3m 5s	remaining: 17m 28s
2000:	learn: 0.9470512	test: 0.9292200	best: 0.9292221 (1999)	total: 4m 9s	remaining: 16m 38s
2500:	learn: 0.9512725	test: 0.9299012	best: 0.9299012 (2500)	total: 5m 13s	remaining: 15m 38s
3000:	learn: 0.9550363	test: 0.9303422	best: 0.9303449 (2997)	total: 6m 14s	remaining: 14m 34s
3500:	learn: 0.9580951	test: 0.9304601	best: 0.9304850 (3410)	total: 7m 17s	remaining: 13m 31s
4000:	learn: 0.9606706	test: 0.9305094	best: 0.9305094 (4000)	total: 8m 16s	remaining: 12m 24s
4500:	learn: 0.9628955	test: 0.9304956	best: 0.9305381 (4316)	total: 9m 14s	remaining: 11m 17s
Stopped by overfitting detector  (500 iterations wait)

bestTest

In [4]:
!ls -lh /kaggle/working

total 1.1M
drwxr-xr-x 5 root root 4.0K Jun  1 09:17 catboost_info
---------- 1 root root  28K Jun  1 10:12 __notebook__.ipynb
-rw-r--r-- 1 root root 1.1M Jun  1 10:12 submission.csv
